# metric-flux: dissipation / energy-flux accountant test

Checkpoint **§L / action L1** (`Docs/checkpoint 3 July 2026.md`).

EvoCA is a dissipative structure: food (free energy) is injected each step
(**source** = `food_inc` over `env_mask` cells) and destroyed through three
**sinks**:

- **(a) tax** actually removed from private food (Phase 2c)
- **(b) death** — the remaining private food of a cell killed by eviction (Phase 4)
- **(c) overflow** — `food_inc` discarded past the `F=1` regeneration cap (Phase 2)

Eating (`mouthful`) is an internal `F_food -> f_priv` transfer, not a sink
(the intake side, already summed as `g_sum_mouthful`).

Whole-system energy identity, checked below every step:

$$\text{source\_step} = \Delta\,\text{storage} + \text{dissipation\_step}$$

where `storage = ΣF_food + Σf_priv` and
`dissipation_step = tax_step + death_step + overflow_step`.

Per-capita throughput `phi = diss_step / alive` is Chaisson's energy-rate-density
and is the principled x-axis material for the §B `food_inc`/`tax` sweep.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'python'))
import numpy as np
from evoca_py import EvoCA

# --- a sustained run that exercises all three sinks -----------------
N = 96
sim = EvoCA()
sim.set_seed(3)
sim.init(N=N, food_inc=0.03, m_scale=2.0, tax=0.004,
         mu_lut=0.002, mu_egenome=0.02, gdiff=0.3)
sim.set_lut_random(2)
sim.set_egenome_random()
sim.set_f_all(0.5)   # seed private food so cells survive the first tax
sim.set_F_all(0.5)
print('init done; N =', N)

init done; N = 96


In [2]:
# --- run and check the per-step energy identity every step ----------
N_STEPS = 500
# float32 lattice of N*N cells: rounding noise scales ~ N*N*eps.
TOL = 1e-2

d_prev = sim.get_dissipation()
max_err = 0.0
worst = None
for step in range(N_STEPS):
    sim.step()
    d = sim.get_dissipation()
    d_storage = d['storage'] - d_prev['storage']
    lhs = d['source_step']
    rhs = d_storage + d['diss_step']
    err = abs(lhs - rhs)
    if err > max_err:
        max_err = err
        worst = (step, lhs, rhs, d['alive'])
    d_prev = d

print(f'per-step identity  source == d(storage) + dissipation')
print(f'  max abs error over {N_STEPS} steps: {max_err:.3e}')
print(f'  worst step (step, lhs, rhs, alive): {worst}')
identity_ok = max_err < TOL
print('  IDENTITY', 'PASS' if identity_ok else 'FAIL',
      f'(tol={TOL})')

per-step identity  source == d(storage) + dissipation
  max abs error over 500 steps: 3.679e-04
  worst step (step, lhs, rhs, alive): (0, 276.47999382019043, 276.4796259701252, 9216)
  IDENTITY PASS (tol=0.01)


In [3]:
# --- print the full dissipation dict --------------------------------
d = sim.get_dissipation()
print('dissipation dict:')
for k, v in d.items():
    print(f'  {k:16s} {v}')

dissipation dict:
  source_step      276.47999382019043
  diss_step        282.5894503593445
  tax_step         36.864001750946045
  death_step       245.72544860839844
  overflow_step    0.0
  source_cum       138239.99691009521
  diss_cum         140094.1300907135
  tax_cum          18432.000875473022
  death_cum        121662.12921524048
  overflow_cum     0.0
  alive            9216
  mouthful_step    278.1961264862912
  phi              0.03066291779072748
  storage_env      666.1545111052692
  storage_priv     6695.681163907051
  storage          7361.83567501232
  step             500


In [4]:
# --- cumulative sanity: source_cum == diss_cum + final storage ------
# Every unit of food ever injected is now EITHER still stored in the
# lattice OR was dissipated through one of the three sinks. Initial
# storage (the seeded f/F) offsets the balance, so account for it.
init_storage = N * N * (0.5 + 0.5)   # set_f_all(0.5) + set_F_all(0.5)

lhs = d['source_cum'] + init_storage
rhs = d['diss_cum'] + d['storage']
err = abs(lhs - rhs)
rel = err / max(lhs, 1.0)
print('cumulative balance:  source_cum + init_storage == diss_cum + storage')
print(f'  source_cum       = {d["source_cum"]:.3f}')
print(f'  + init_storage   = {init_storage:.3f}')
print(f'  = lhs            = {lhs:.3f}')
print(f'  diss_cum         = {d["diss_cum"]:.3f}')
print(f'  + final storage  = {d["storage"]:.3f}')
print(f'  = rhs            = {rhs:.3f}')
print(f'  abs err = {err:.3e}   rel err = {rel:.3e}')
cum_ok = rel < 1e-4
print('  CUMULATIVE', 'PASS' if cum_ok else 'FAIL', '(rel tol=1e-4)')

# sinks decomposition should sum to diss_cum
sink_sum = d['tax_cum'] + d['death_cum'] + d['overflow_cum']
decomp_ok = abs(sink_sum - d['diss_cum']) < 1e-3
print(f'  tax+death+overflow = {sink_sum:.3f}  vs diss_cum = {d["diss_cum"]:.3f}',
      '-> DECOMP', 'PASS' if decomp_ok else 'FAIL')
print(f'  all three sinks exercised: tax={d["tax_cum"]>0} '
      f'death={d["death_cum"]>0} overflow={d["overflow_cum"]>0}')

cumulative balance:  source_cum + init_storage == diss_cum + storage
  source_cum       = 138239.997
  + init_storage   = 9216.000
  = lhs            = 147455.997
  diss_cum         = 140094.130
  + final storage  = 7361.836
  = rhs            = 147455.966
  abs err = 3.114e-02   rel err = 2.112e-07
  CUMULATIVE PASS (rel tol=1e-4)
  tax+death+overflow = 140094.130  vs diss_cum = 140094.130 -> DECOMP PASS
  all three sinks exercised: tax=True death=True overflow=False


In [5]:
# --- second regime: exercise the overflow sink (c) ------------------
# A near-empty world with strong regeneration and no eating fills
# F_food to the F=1 cap, so subsequent food_inc overflows and is lost.
sim2 = EvoCA()
sim2.set_seed(7)
sim2.init(N=48, food_inc=0.2, m_scale=0.0, tax=0.0)
sim2.set_alive_fraction(0.02)   # sparse population -> most cells regen freely
d2_prev = sim2.get_dissipation()
max_err2 = 0.0
for step in range(50):
    sim2.step()
    d2 = sim2.get_dissipation()
    rhs = (d2['storage'] - d2_prev['storage']) + d2['diss_step']
    max_err2 = max(max_err2, abs(d2['source_step'] - rhs))
    d2_prev = d2
d2 = sim2.get_dissipation()
overflow_ok = d2['overflow_cum'] > 0 and max_err2 < 1e-2
print(f'overflow regime: overflow_cum={d2["overflow_cum"]:.2f} '
      f'max identity err={max_err2:.3e} -> OVERFLOW',
      'PASS' if overflow_ok else 'FAIL')

# --- overall PASS/FAIL ---------------------------------------------
all_ok = identity_ok and cum_ok and decomp_ok and overflow_ok
print('=' * 48)
print('metric-flux dissipation accountant:',
      'PASS' if all_ok else 'FAIL')
print('=' * 48)
assert all_ok

overflow regime: overflow_cum=20736.00 max identity err=1.030e-04 -> OVERFLOW PASS
metric-flux dissipation accountant: PASS
